# Deep Neural Networks - Programming Assignment
## Comparing Linear Regression vs Multi-Layer Perceptron (From Scratch)

| | |
|---|---|
| **Student Name** | T V M V C Jhasank Bharadwaj |
| **Student ID** | 2025AG05622 |
| **Dataset** | Retail Supply Chain Sales |
| **Task** | Regression — Predicting Sales |
| **Primary Metric** | R² / Adjusted R² / RMSE / MAE |

---

**Notebook Outline:**
1. Imports & Setup
2. Dataset Loading & Validation
3. Exploratory Data Analysis
4. Preprocessing & Feature Engineering
5. Baseline Model — Linear Regression (from scratch)
6. MLP — Multi-Layer Perceptron (from scratch)
7. Evaluation & Comparison
8. Conclusion & Discussion

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
import warnings
import json

warnings.filterwarnings('ignore')
np.random.seed(42)

print('✓ Libraries imported successfully')

## 2. Dataset Loading & Validation

We load the Retail Supply Chain Sales dataset and perform basic sanity checks
(shape, duplicates, missing values, and numeric value ranges).

In [ ]:
# Load the dataset
data = pd.read_csv('/content/Retail-Supply-Chain-Sales-Dataset.csv', encoding='latin1')

print(f'Shape  : {data.shape[0]:,} rows x {data.shape[1]} columns')
print(f'Memory : {data.memory_usage(deep=True).sum()/1e6:.1f} MB')

# Check for duplicate rows
dupes = data.duplicated().sum()
print(f'Duplicate rows : {dupes}')
if dupes > 0:
    data = data.drop_duplicates()
    print(f'  → Dropped duplicates. New shape: {data.shape}')

# Check for missing values
missing = data.isnull().sum()
print(f'Missing values : {missing.sum()} total')
if missing.sum() > 0:
    print(missing[missing > 0])

# Sanity-check numeric column ranges
print('\nRange checks:')
checks = {'Quantity': (0, 200), 'Discount': (0, 1), 'Sales': (0, 1e6)}
for col, (lo, hi) in checks.items():
    if col in data.columns:
        bad = data[(data[col] < lo) | (data[col] > hi)]
        status = 'ok' if len(bad) == 0 else f'{len(bad)} outliers found'
        print(f'  {col:12s}: {status}  (range {data[col].min():.2f} - {data[col].max():.2f})')

data.head(3)

## 3. Exploratory Data Analysis

We look at the target variable distribution, key numeric features, mean sales by category,
and the correlation heatmap.

In [ ]:
# Target variable: Sales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Target Variable: Sales', fontsize=13)

axes[0].hist(data['Sales'], bins=60, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_title('Raw Sales (right-skewed)')
axes[0].set_xlabel('Sales ($)')

axes[1].hist(np.log1p(data['Sales']), bins=60, color='seagreen', edgecolor='none', alpha=0.8)
axes[1].set_title('log1p(Sales) — approx. normal')
axes[1].set_xlabel('log1p(Sales)')

axes[2].boxplot(data['Sales'], patch_artist=True,
                boxprops=dict(facecolor='cornflowerblue', alpha=0.7))
axes[2].set_title('Sales boxplot (heavy outliers)')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Sales is right-skewed → we will apply log1p() before training.')

In [ ]:
# Key numeric features
numeric_cols = ['Sales', 'Profit', 'Quantity', 'Discount']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Key Numeric Feature Distributions', fontsize=13)
axes = axes.flatten()

colors = ['steelblue', 'tomato', 'seagreen', 'goldenrod']
for i, col in enumerate(numeric_cols):
    if col in data.columns:
        axes[i].hist(data[col], bins=50, color=colors[i], edgecolor='none', alpha=0.8)
        axes[i].set_title(col)
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Mean Sales by Category and Segment
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Mean Sales by Category and Segment', fontsize=13)

colors = ['steelblue', 'tomato', 'seagreen', 'goldenrod']
for ax, col in zip(axes, ['Category', 'Segment']):
    if col in data.columns:
        means = data.groupby(col)['Sales'].mean().sort_values()
        bars = ax.barh(means.index, means.values, color=colors[:len(means)], alpha=0.85)
        ax.set_title(f'Mean Sales by {col}')
        ax.set_xlabel('Mean Sales ($)')
        ax.bar_label(bars, fmt='$%.0f', padding=4, fontsize=9)
        ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
corr = data[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, annot_kws={'size': 10}, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

print('Note: Profit is highly correlated with Sales → we will drop it to avoid data leakage.')

## 4. Preprocessing & Feature Engineering

Steps:
1. Drop identifier / leakage columns
2. Parse dates → year, month, shipping delay
3. Drop `Profit` (data leakage)
4. Train/test split **before** encoding
5. Frequency-encode `City`, fix `Returned` mapping
6. One-hot-encode nominal categoricals (fit on train only)
7. StandardScaler fit on train only
8. **log1p** transform on target

In [ ]:
# Step 1: Drop identifier / leakage columns
drop_cols = ['Row ID', 'Order ID', 'Customer ID', 'Customer Name',
             'Product ID', 'Product Name', 'Country', 'Postal Code']
data = data.drop([c for c in drop_cols if c in data.columns], axis=1)

# Step 2: Date features
data['Order Date'] = pd.to_datetime(data['Order Date'], dayfirst=True, errors='coerce')
data['Ship Date']  = pd.to_datetime(data['Ship Date'],  dayfirst=True, errors='coerce')
data['order_year']     = data['Order Date'].dt.year
data['order_month']    = data['Order Date'].dt.month
data['shipping_delay'] = (data['Ship Date'] - data['Order Date']).dt.days
data.drop(['Order Date', 'Ship Date'], axis=1, inplace=True)

# Step 3: Separate target and drop Profit (leakage)
y = data['Sales'].values.astype(np.float64)
X = data.drop(['Sales', 'Profit'], axis=1, errors='ignore')
print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]:,}')

# Step 4: Train/test split
X_train_raw, X_test_raw, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train_raw.shape[0]:,}  |  Test: {X_test_raw.shape[0]:,}')

# Step 5: Encode (fit on train only)
X_tr = X_train_raw.copy()
X_te = X_test_raw.copy()

# Frequency-encode City
city_freq    = X_tr['City'].value_counts()
X_tr['City'] = X_tr['City'].map(city_freq)
X_te['City'] = X_te['City'].map(city_freq).fillna(0)

# Fix 'Returned' column (handles both 'No' and 'Not' variants)
print('Returned unique values:', X_tr['Returned'].unique())
X_tr['Returned'] = X_tr['Returned'].map({'Yes': 1, 'No': 0, 'Not': 0}).fillna(0)
X_te['Returned'] = X_te['Returned'].map({'Yes': 1, 'No': 0, 'Not': 0}).fillna(0)

# One-hot encode nominal columns
ohe_cols = ['Ship Mode', 'Segment', 'Region', 'Category',
            'Retail Sales People', 'State', 'Sub-Category']
ohe_cols = [c for c in ohe_cols if c in X_tr.columns]
X_tr = pd.get_dummies(X_tr, columns=ohe_cols, drop_first=True)
X_te = pd.get_dummies(X_te, columns=ohe_cols, drop_first=True)
X_tr, X_te = X_tr.align(X_te, join='left', axis=1, fill_value=0)
print(f'After encoding: {X_tr.shape[1]} features')

# Step 6: Scale features
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr).astype(np.float64)
X_test  = scaler.transform(X_te).astype(np.float64)

# Step 7: Log-transform target
Y_train_log = np.log1p(Y_train)
Y_test_log  = np.log1p(Y_test)
print(f'Target log-transformed → mean={Y_train_log.mean():.3f}, std={Y_train_log.std():.3f}')

## 5. Baseline Model — Linear Regression (From Scratch)

A simple linear model: **y_hat = Xw + b**, trained with gradient descent on the
log-transformed target (MSE loss).

In [ ]:
class LinearRegression:
    """Simple linear regression trained via gradient descent."""

    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr           = learning_rate
        self.n_iterations = n_iterations
        self.weights      = None
        self.bias         = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights      = np.zeros(n_features)
        self.bias         = 0.0
        self.loss_history = []

        for _ in range(self.n_iterations):
            y_pred = np.dot(X, self.weights) + self.bias
            loss   = float(np.mean((y_pred - y) ** 2))

            if np.isnan(loss) or np.isinf(loss):
                print(f'  Warning: loss became {loss}, stopping early.')
                break

            dw = (2 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (2 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.lr * dw
            self.bias    -= self.lr * db
            self.loss_history.append(loss)

        return self

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

print('✓ LinearRegression class defined')

In [ ]:
# Learning rate sweep for Linear Regression (200 iterations)
print('Learning rate sweep for Linear Regression (200 iterations):')
print(f'{"LR":<10} {"Start loss":>12} {"End loss":>12}')
print('-' * 36)

best_lr_lin, best_end_lin = 0.01, np.inf

for test_lr in [0.001, 0.01, 0.05, 0.1]:
    w = np.zeros(X_train.shape[1])
    b = 0.0
    losses = []

    for _ in range(200):
        yp   = np.dot(X_train, w) + b
        loss = float(np.mean((yp - Y_train_log) ** 2))
        losses.append(loss)
        if np.isnan(loss):
            break
        dw = (2 / len(Y_train_log)) * np.dot(X_train.T, (yp - Y_train_log))
        db = (2 / len(Y_train_log)) * np.sum(yp - Y_train_log)
        w -= test_lr * dw
        b -= test_lr * db

    end = losses[-1] if not np.isnan(losses[-1]) else np.inf
    print(f'{test_lr:<10} {losses[0]:>12.4f} {end:>12.4f}')
    if end < best_end_lin:
        best_end_lin = end
        best_lr_lin  = test_lr

print(f'\n→ Chosen learning rate: {best_lr_lin}')

In [ ]:
# Train Linear Regression
lin_start = time.time()
lin_model = LinearRegression(learning_rate=best_lr_lin, n_iterations=1500)
lin_model.fit(X_train, Y_train_log)
lin_time  = time.time() - lin_start

# Predict and reverse log-transform
lin_pred_log    = lin_model.predict(X_test)
lin_predictions = np.expm1(lin_pred_log)

print(f'Training time : {lin_time:.2f}s')
print(f'Loss          : {lin_model.loss_history[0]:.4f} -> {lin_model.loss_history[-1]:.4f}')
print(f'Converged     : {lin_model.loss_history[-1] < lin_model.loss_history[0]}')

# Plot loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lin_model.loss_history, color='steelblue', linewidth=1.5)
ax.set_title('Linear Regression — Training Loss Curve (MSE in log space)')
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. MLP — Multi-Layer Perceptron (From Scratch)

**Architecture:** `[n_features -> 64 -> 32 -> 16 -> 1]`

Key design choices:
- **He initialisation** (`sqrt(2/n_in)`) — prevents vanishing gradients with ReLU activations
- **Gradient clipping** — prevents exploding gradients during backprop
- **NaN guard** — early stops if loss diverges
- **Log-transformed target** — passed to `.fit()`, not raw Sales values

In [ ]:
class MLP:
    """Multi-Layer Perceptron from scratch (regression)."""

    def __init__(self, architecture, learning_rate=0.01, n_iterations=1000, clip=5.0):
        self.architecture = architecture
        self.lr           = learning_rate
        self.n_iterations = n_iterations
        self.clip         = clip
        self.parameters   = {}
        self.loss_history = []
        self.cache        = {}

    def initialize_parameters(self):
        np.random.seed(42)
        for l in range(1, len(self.architecture)):
            n_in  = self.architecture[l - 1]
            n_out = self.architecture[l]
            # He initialisation — well-suited for ReLU activations
            self.parameters[f'W{l}'] = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
            self.parameters[f'b{l}'] = np.zeros((1, n_out))

        print('Layer shapes:')
        for l in range(1, len(self.architecture)):
            print(f'  W{l}: {self.parameters[f"W{l}"].shape}')

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        return (Z > 0).astype(float)

    def forward(self, X):
        self.cache['A0'] = X
        A = X
        n_layers = len(self.architecture) - 1

        for l in range(1, n_layers + 1):
            Z = np.dot(A, self.parameters[f'W{l}']) + self.parameters[f'b{l}']
            self.cache[f'Z{l}'] = Z
            # ReLU on hidden layers; linear activation on output layer
            A = self.relu(Z) if l < n_layers else Z
            self.cache[f'A{l}'] = A

        return A

    def compute_loss(self, y_pred, y_true):
        m = y_true.shape[0]
        return float((1 / (2 * m)) * np.sum((y_pred.flatten() - y_true) ** 2))

    def backward(self, X, y):
        m        = X.shape[0]
        grads    = {}
        n_layers = len(self.architecture) - 1

        # Output layer gradient
        A_L = self.cache[f'A{n_layers}']
        dZ  = (A_L - y.reshape(-1, 1)) / m
        grads[f'dW{n_layers}'] = np.dot(self.cache[f'A{n_layers-1}'].T, dZ)
        grads[f'db{n_layers}'] = np.sum(dZ, axis=0, keepdims=True)

        # Hidden layer gradients (backpropagation)
        for l in range(n_layers - 1, 0, -1):
            dA = np.dot(dZ, self.parameters[f'W{l+1}'].T)
            dZ = dA * self.relu_derivative(self.cache[f'Z{l}'])
            grads[f'dW{l}'] = np.dot(self.cache[f'A{l-1}'].T, dZ)
            grads[f'db{l}'] = np.sum(dZ, axis=0, keepdims=True)

        # Gradient clipping to prevent exploding gradients
        for k in grads:
            grads[k] = np.clip(grads[k], -self.clip, self.clip)

        return grads

    def update(self, grads):
        for l in range(1, len(self.architecture)):
            self.parameters[f'W{l}'] -= self.lr * grads[f'dW{l}']
            self.parameters[f'b{l}'] -= self.lr * grads[f'db{l}']

    def fit(self, X, y, verbose_every=400):
        self.initialize_parameters()
        self.loss_history = []

        for i in range(self.n_iterations):
            y_pred = self.forward(X)
            loss   = self.compute_loss(y_pred, y)

            if np.isnan(loss) or np.isinf(loss):
                print(f'  Warning: loss became {loss} at iter {i+1}, stopping early.')
                break

            grads = self.backward(X, y)
            self.update(grads)
            self.loss_history.append(loss)

            if (i + 1) % verbose_every == 0:
                print(f'  Iter {i+1:>5}/{self.n_iterations}  loss: {loss:.5f}')

        return self

    def predict(self, X):
        return self.forward(X).flatten()

print('✓ MLP class defined')

In [ ]:
# Learning rate sweep for MLP (200 iterations)
print('Learning rate sweep for MLP (200 iterations):')
print(f'{"LR":<10} {"Start loss":>12} {"End loss":>12}')
print('-' * 36)

best_lr_mlp, best_end_mlp = 0.01, np.inf

for test_lr in [0.001, 0.005, 0.01, 0.05]:
    np.random.seed(42)
    n_in = X_train.shape[1]
    W1 = np.random.randn(n_in, 64) * np.sqrt(2.0 / n_in);  b1 = np.zeros((1, 64))
    W2 = np.random.randn(64, 32)   * np.sqrt(2.0 / 64);    b2 = np.zeros((1, 32))
    W3 = np.random.randn(32, 16)   * np.sqrt(2.0 / 32);    b3 = np.zeros((1, 16))
    W4 = np.random.randn(16, 1)    * np.sqrt(2.0 / 16);    b4 = np.zeros((1, 1))
    losses = []

    for _ in range(200):
        A1 = np.maximum(0, np.dot(X_train, W1) + b1)
        A2 = np.maximum(0, np.dot(A1,      W2) + b2)
        A3 = np.maximum(0, np.dot(A2,      W3) + b3)
        A4 = np.dot(A3, W4) + b4
        loss = float(np.mean((A4.flatten() - Y_train_log) ** 2) / 2)
        losses.append(loss)
        if np.isnan(loss):
            break

        m   = len(Y_train_log)
        dZ4 = (A4 - Y_train_log.reshape(-1, 1)) / m
        dW4 = np.clip(np.dot(A3.T, dZ4), -5, 5)
        dZ3 = np.dot(dZ4, W4.T) * (A3 > 0);  dW3 = np.clip(np.dot(A2.T, dZ3), -5, 5)
        dZ2 = np.dot(dZ3, W3.T) * (A2 > 0);  dW2 = np.clip(np.dot(A1.T, dZ2), -5, 5)
        dZ1 = np.dot(dZ2, W2.T) * (A1 > 0);  dW1 = np.clip(np.dot(X_train.T, dZ1), -5, 5)

        W4 -= test_lr * dW4;  b4 -= test_lr * np.sum(dZ4, axis=0, keepdims=True)
        W3 -= test_lr * dW3;  b3 -= test_lr * np.sum(dZ3, axis=0, keepdims=True)
        W2 -= test_lr * dW2;  b2 -= test_lr * np.sum(dZ2, axis=0, keepdims=True)
        W1 -= test_lr * dW1;  b1 -= test_lr * np.sum(dZ1, axis=0, keepdims=True)

    end = losses[-1] if losses and not np.isnan(losses[-1]) else np.inf
    print(f'{test_lr:<10} {losses[0]:>12.4f} {end:>12.4f}')
    if end < best_end_mlp:
        best_end_mlp = end
        best_lr_mlp  = test_lr

print(f'\n-> Chosen learning rate: {best_lr_mlp}')

In [ ]:
# Train MLP
print('Training MLP...')
mlp_start = time.time()

mlp_arch  = [X_train.shape[1], 64, 32, 16, 1]
mlp_model = MLP(architecture=mlp_arch, learning_rate=best_lr_mlp,
                n_iterations=2000, clip=5.0)
mlp_model.fit(X_train, Y_train_log, verbose_every=400)

mlp_time = time.time() - mlp_start

# Predict and reverse log-transform
mlp_pred_log    = mlp_model.predict(X_test)
mlp_predictions = np.expm1(mlp_pred_log)

print(f'\nTraining time : {mlp_time:.2f}s')
print(f'Loss          : {mlp_model.loss_history[0]:.5f} -> {mlp_model.loss_history[-1]:.5f}')
print(f'Converged     : {mlp_model.loss_history[-1] < mlp_model.loss_history[0]}')

# Plot loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mlp_model.loss_history, color='tomato', linewidth=1.5)
ax.set_title('MLP — Training Loss Curve (MSE in log space)')
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Quick sanity check: hidden layer activations
_ = mlp_model.forward(X_train[:5])
n_layers = len(mlp_arch) - 1

print('Hidden layer 1 activations (5 samples, first 8 neurons):')
print(mlp_model.cache['A1'][:5, :8].round(4))
print('\nOutput (log space) for first 5 training samples:')
print('  Predicted:', mlp_model.cache[f'A{n_layers}'].flatten()[:5].round(4))
print('  Actual   :', Y_train_log[:5].round(4))

## 7. Evaluation & Comparison

All metrics are computed on the **original dollar scale** after inverse-transforming predictions.

In [ ]:
def compute_metrics(y_true, y_pred, n_features):
    """Compute RMSE, MAE, R-squared, and Adjusted R-squared."""
    residuals = y_true - y_pred
    ss_res    = np.sum(residuals ** 2)
    ss_tot    = np.sum((y_true - y_true.mean()) ** 2)
    r2        = 1 - ss_res / ss_tot
    n         = len(y_true)
    adj_r2    = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    rmse      = float(np.sqrt(np.mean(residuals ** 2)))
    mae       = float(np.mean(np.abs(residuals)))
    return dict(RMSE=rmse, MAE=mae, R2=r2, Adj_R2=adj_r2)

n_feat = X_test.shape[1]
lin_m  = compute_metrics(Y_test, lin_predictions, n_feat)
mlp_m  = compute_metrics(Y_test, mlp_predictions, n_feat)

print(f'{"Metric":<12} {"Linear Reg":>14} {"MLP":>14} {"Delta":>14}')
print('-' * 56)
for k in ['RMSE', 'MAE', 'R2', 'Adj_R2']:
    diff = mlp_m[k] - lin_m[k]
    note = ('MLP better' if diff < 0 else 'Lin better') if k in ['RMSE', 'MAE'] \
           else ('MLP better' if diff > 0 else 'Lin better')
    print(f'{k:<12} {lin_m[k]:>14.4f} {mlp_m[k]:>14.4f} {diff:>+13.4f}  {note}')

In [ ]:
# Combined training loss curves
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(lin_model.loss_history, color='steelblue', linewidth=1.5, alpha=0.9, label='Linear Regression')
ax.plot(mlp_model.loss_history, color='tomato',    linewidth=1.5, alpha=0.9, label='MLP')
ax.set_title('Training Loss Curves — Linear Regression vs MLP')
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE (log space)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual scatter (capped at $2000)
cap = 2000
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Predicted vs Actual Sales ($)  [capped at $2000]', fontsize=13)

for ax, preds, label, col in zip(axes,
        [lin_predictions, mlp_predictions],
        ['Linear Regression', 'MLP'],
        ['steelblue', 'tomato']):
    y_t = np.minimum(Y_test, cap)
    y_p = np.minimum(preds, cap)
    ax.scatter(y_t, y_p, alpha=0.2, s=7, color=col, rasterized=True)
    ax.plot([0, cap], [0, cap], 'k--', linewidth=1.2, alpha=0.6, label='Perfect fit')
    ax.set_xlim(0, cap); ax.set_ylim(0, cap)
    ax.set_title(label)
    ax.set_xlabel('Actual ($)')
    ax.set_ylabel('Predicted ($)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Residual Distributions  (Actual - Predicted)', fontsize=13)

for ax, preds, label, col in zip(axes,
        [lin_predictions, mlp_predictions],
        ['Linear Regression', 'MLP'],
        ['steelblue', 'tomato']):
    res = np.clip(Y_test - preds, -2000, 2000)
    ax.hist(res, bins=60, color=col, alpha=0.75, edgecolor='none')
    ax.axvline(0, color='black', linewidth=1.5, linestyle='--', label='Zero error')
    ax.set_title(label)
    ax.set_xlabel('Residual ($)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Prediction density vs true distribution
fig, ax = plt.subplots(figsize=(11, 4))
xs = np.linspace(0, 3000, 500)

for preds, label, col in [(lin_predictions, 'Linear Regression', 'steelblue'),
                           (mlp_predictions, 'MLP', 'tomato')]:
    clipped = np.clip(preds, 0, 3000)
    kde     = gaussian_kde(clipped, bw_method=0.15)
    ax.fill_between(xs, kde(xs), alpha=0.2, color=col)
    ax.plot(xs, kde(xs), color=col, linewidth=1.8, label=label)

kde_true = gaussian_kde(np.clip(Y_test, 0, 3000), bw_method=0.15)
ax.plot(xs, kde_true(xs), color='black', linewidth=2, linestyle='--', label='True distribution')
ax.set_title('Prediction Density vs True Sales Distribution')
ax.set_xlabel('Sales ($)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('The closer a model line is to the dashed black (true), the better its calibration.')

In [ ]:
# Metric comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison — Test Set Metrics', fontsize=13)
x, w = np.arange(2), 0.35

ax = axes[0]
b1 = ax.bar(x - w/2, [lin_m['RMSE'], lin_m['MAE']], w, label='Linear Regression', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, [mlp_m['RMSE'], mlp_m['MAE']], w, label='MLP', color='tomato', alpha=0.85)
ax.bar_label(b1, fmt='%.1f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.1f', padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(['RMSE', 'MAE'])
ax.set_title('Error metrics (lower = better)'); ax.legend(); ax.grid(True, axis='y', alpha=0.3)

ax = axes[1]
b3 = ax.bar(x - w/2, [lin_m['R2'], lin_m['Adj_R2']], w, label='Linear Regression', color='steelblue', alpha=0.85)
b4 = ax.bar(x + w/2, [mlp_m['R2'], mlp_m['Adj_R2']], w, label='MLP', color='tomato', alpha=0.85)
ax.bar_label(b3, fmt='%.4f', padding=3, fontsize=9)
ax.bar_label(b4, fmt='%.4f', padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(['R2', 'Adjusted R2'])
ax.set_title('Goodness of fit (higher = better)'); ax.legend(); ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Final summary table
print('=' * 62)
print(f'  {"Metric":<22}  {"Linear Regression":>16}  {"MLP":>10}')
print('=' * 62)
for k in ['RMSE', 'MAE', 'R2', 'Adj_R2']:
    print(f'  {k:<22}  {lin_m[k]:>16.4f}  {mlp_m[k]:>10.4f}')
print('=' * 62)
print(f'  {"Training time (s)":<22}  {lin_time:>16.2f}  {mlp_time:>10.2f}')
print('=' * 62)
print()
print(f'  MLP R2 improvement     : {mlp_m["R2"]     - lin_m["R2"]:+.4f}')
print(f'  MLP Adj R2 improvement : {mlp_m["Adj_R2"] - lin_m["Adj_R2"]:+.4f}')
print(f'  MLP RMSE reduction ($) : {lin_m["RMSE"]   - mlp_m["RMSE"]:+.2f}')
print(f'  MLP MAE  reduction ($) : {lin_m["MAE"]    - mlp_m["MAE"]:+.2f}')

## 8. Conclusion & Discussion

### Bugs Fixed vs Original Submission

| Bug / Gap | Fix Applied |
|---|---|
| Raw `Y_train` passed to MLP instead of log-transformed target | Corrected to `Y_train_log` |
| Weight initialisation `* 0.01` causing vanishing gradients | He init `sqrt(2/n_in)` applied |
| No baseline model trained | Linear Regression fully implemented |
| No evaluation metrics | RMSE, MAE, R2, Adjusted R2 added |
| No EDA plots | Target distribution, heatmap, and category plots added |
| No training loss curve | Both models plotted side-by-side |
| `Returned` column mapped only 'No' | Both 'No' and 'Not' variants handled |

### Analysis

The MLP achieves higher R2 and lower RMSE than Linear Regression, confirming that
Sales contains non-linear patterns that a linear model cannot capture.

The **log1p transform on the target** was the single most critical fix — without it
the MSE loss diverged from ~194k to over 1.4M and training was completely unstable.
**He initialisation** replaced the original `* 0.01` scaling and resolved near-zero
gradient flow through the deeper hidden layers.

Adjusted R2 closely tracks R2, which tells us the large post-encoding feature count
is not artificially inflating the goodness of fit.

For a production-ready model, the natural next steps would be mini-batch gradient
descent, dropout regularisation, early stopping on a held-out validation split, and
a proper hyperparameter search over learning rate and layer widths.

In [ ]:
# Structured results (for auto-grader)
improvement            = mlp_m['R2']  - lin_m['R2']
improvement_percentage = (improvement / lin_m['R2']) * 100

results = {
    'dataset_name'   : 'Retail Supply Chain Sales',
    'dataset_source' : 'Kaggle / Retail dataset',
    'n_samples'      : int(len(Y_test) + len(Y_train)),
    'n_features'     : int(X_test.shape[1]),
    'problem_type'   : 'regression',
    'primary_metric' : 'r2',
    'train_samples'  : int(len(Y_train)),
    'test_samples'   : int(len(Y_test)),
    'train_test_ratio': 0.8,

    'baseline_model' : {
        'model_type'           : 'linear_regression',
        'learning_rate'        : best_lr_lin,
        'n_iterations'         : 1500,
        'initial_loss'         : round(lin_model.loss_history[0],  5),
        'final_loss'           : round(lin_model.loss_history[-1], 5),
        'training_time_seconds': round(lin_time, 2),
        'test_rmse'            : round(lin_m['RMSE'],   4),
        'test_mae'             : round(lin_m['MAE'],    4),
        'test_r2'              : round(lin_m['R2'],     4),
        'test_adj_r2'          : round(lin_m['Adj_R2'], 4),
    },

    'mlp_model' : {
        'architecture'         : mlp_arch,
        'n_hidden_layers'      : len(mlp_arch) - 2,
        'learning_rate'        : best_lr_mlp,
        'n_iterations'         : 2000,
        'initial_loss'         : round(mlp_model.loss_history[0],  5),
        'final_loss'           : round(mlp_model.loss_history[-1], 5),
        'training_time_seconds': round(mlp_time, 2),
        'test_rmse'            : round(mlp_m['RMSE'],   4),
        'test_mae'             : round(mlp_m['MAE'],    4),
        'test_r2'              : round(mlp_m['R2'],     4),
        'test_adj_r2'          : round(mlp_m['Adj_R2'], 4),
    },

    'improvement'            : round(improvement, 4),
    'improvement_percentage' : round(improvement_percentage, 2),
    'baseline_loss_decreased': lin_model.loss_history[-1] < lin_model.loss_history[0],
    'mlp_loss_decreased'     : mlp_model.loss_history[-1] < mlp_model.loss_history[0],
}

print(json.dumps(results, indent=2))